# 12. 표준 라이브러리 실무 팁

매번 검색하게 되는 `pathlib`, `collections`, `itertools`, `functools`, `datetime`, `logging`을 모아놓은 치트시트.

## pathlib: 문자열 경로 조작 대신 쓰기

`os.path.join`보다 가독성이 좋고, 경로 관련 메서드가 객체에 다 붙어 있습니다.

In [1]:
from pathlib import Path

p = Path("/home/claude/knowledge/python-syntax/12-stdlib-tips.ipynb")

print(p.name)        # 파일명
print(p.stem)        # 확장자 제외 파일명
print(p.suffix)       # 확장자
print(p.parent)       # 상위 폴더
print(p.parent / "README.md")  # '/' 연산자로 경로 합치기
print(p.exists())

for child in p.parent.glob("*.ipynb"):  # 패턴 매칭으로 파일 찾기
    print(" -", child.name)


12-stdlib-tips.ipynb
12-stdlib-tips
.ipynb
/home/claude/knowledge/python-syntax
/home/claude/knowledge/python-syntax/README.md
True
 - 01-variables-types.ipynb
 - 10-modules-packages.ipynb
 - 03-data-structures.ipynb
 - 11-concurrency-async.ipynb
 - 08-exceptions-context-manager.ipynb
 - 09-type-hints.ipynb
 - 12-stdlib-tips.ipynb
 - 07-iterators-generators.ipynb
 - 04-functions-advanced.ipynb
 - 02-control-flow.ipynb
 - 05-decorators.ipynb
 - 06-oop-classes.ipynb


## collections: Counter, defaultdict, namedtuple, deque

In [2]:
from collections import Counter, defaultdict, namedtuple, deque

# Counter: 빈도수 세기
words = ["lidar", "radar", "lidar", "camera", "lidar", "radar"]
print(Counter(words))
print(Counter(words).most_common(2))

# defaultdict: 키가 없을 때 기본값을 자동 생성 (if key not in dict 분기 제거)
groups = defaultdict(list)
for sensor, value in [("lidar", 1), ("radar", 2), ("lidar", 3)]:
    groups[sensor].append(value)
print(dict(groups))

# namedtuple: 필드 이름이 있는 가벼운 튜플 (dataclass 나오기 전부터 쓰이던 방식)
Point = namedtuple("Point", ["x", "y"])
p = Point(1, 2)
print(p.x, p.y, p)

# deque: 양쪽 끝에서 빠르게 추가/제거 (큐/스택, 슬라이딩 윈도우에 적합)
dq = deque(maxlen=3)  # 최근 3개만 유지하는 슬라이딩 윈도우
for i in range(6):
    dq.append(i)
print(dq)


Counter({'lidar': 3, 'radar': 2, 'camera': 1})
[('lidar', 3), ('radar', 2)]
{'lidar': [1, 3], 'radar': [2]}
1 2 Point(x=1, y=2)
deque([3, 4, 5], maxlen=3)


## itertools: product, permutations, groupby

In [3]:
import itertools

# product: 중첩 for문 대신 (하이퍼파라미터 그리드서치에 자주 사용)
for lr, batch in itertools.product([0.001, 0.01], [16, 32]):
    print(f"lr={lr}, batch={batch}")

print("---")
print(list(itertools.permutations([1, 2, 3], 2)))  # 순열

print("---")
data = [("lidar", 1), ("lidar", 2), ("radar", 3), ("radar", 4)]
for key, group in itertools.groupby(data, key=lambda x: x[0]):  # 정렬되어 있어야 의도대로 동작
    print(key, list(group))


lr=0.001, batch=16
lr=0.001, batch=32
lr=0.01, batch=16
lr=0.01, batch=32
---
[(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2)]
---
lidar [('lidar', 1), ('lidar', 2)]
radar [('radar', 3), ('radar', 4)]


## functools: reduce, partial

In [4]:
from functools import reduce, partial

# reduce: 누적 계산 (sum/max처럼 전용 함수가 없을 때)
product_all = reduce(lambda acc, x: acc * x, [1, 2, 3, 4], 1)
print(product_all)

# partial: 함수의 일부 인자를 미리 고정해서 새 함수 만들기
def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)
cube = partial(power, exponent=3)
print(square(5), cube(5))


24
25 125


## datetime: 시간 다루기 기초

In [5]:
from datetime import datetime, timedelta

now = datetime.now()
print(now.strftime("%Y-%m-%d %H:%M:%S"))  # 문자열로 포맷팅

later = now + timedelta(hours=2, minutes=30)
print(later.strftime("%Y-%m-%d %H:%M:%S"))

parsed = datetime.strptime("2026-11-26", "%Y-%m-%d")  # 문자열 -> datetime
print(parsed)


2026-11-26 10:47:59
2026-11-26 13:17:59
2026-11-26 00:00:00


## logging: print() 대신 쓰기

`print`는 끄고 켤 수 없고 어디서 찍힌 건지 알기 어렵습니다. `logging`은 레벨별 제어와 포맷(파일명, 시각 등)이 가능해서 실무 코드에서는 기본입니다.

In [6]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,  # 노트북에서 재실행해도 핸들러가 중복되지 않도록
)
logger = logging.getLogger("demo")

logger.debug("이건 기본 레벨(INFO)보다 낮아서 안 보임")
logger.info("모델 로딩 시작")
logger.warning("배치 사이즈가 커서 메모리 부족 위험")
logger.error("추론 실패")


10:47:59 [INFO] 모델 로딩 시작


10:47:59 [WARNING] 배치 사이즈가 커서 메모리 부족 위험


10:47:59 [ERROR] 추론 실패


## 헷갈렸던 점 / 더 찾아볼 것

- `groupby`는 **입력이 미리 정렬돼 있어야** 의도대로 그룹핑됨 (SQL GROUP BY와 다름) — 자주 하는 실수
- 로깅 레벨은 DEBUG < INFO < WARNING < ERROR < CRITICAL 순서로, `basicConfig(level=...)`보다 낮은 레벨은 출력되지 않음